In [1]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer

from dataprocessing import data, data_noutliers

c:\Users\yzhen\OneDrive\Documents\MSPPM-DA\SP2025\ML\project - education\dataprocessing.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['school_state_region'] = data['school_state'].map(state_to_region)
c:\Users\yzhen\OneDrive\Documents\MSPPM-DA\SP2025\ML\project - education\dataprocessing.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['fully_funded'] = data['fully_funded'].map(binary_map)
c:\Users\yzhen\OneDrive\Documents\MSPPM-DA\SP2025\ML\project - education\dataprocessing.py:61: Set

In [2]:
data_noutliers.shape

(485604, 28)

In [3]:
data.shape

(542224, 26)

In [4]:
data_noutliers['fully_funded'].value_counts()

fully_funded
1    347236
0    138368
Name: count, dtype: int64

In [5]:
data['fully_funded'].value_counts()

fully_funded
1    380906
0    161318
Name: count, dtype: int64

**Baseline with Majority Classifier (Funded = 1)**

In [6]:
features = ['school_metro','poverty_level']

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data[features])

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data['fully_funded'], test_size=0.2, random_state=42,stratify=data['fully_funded'])

y_pred = []
for i in range(len(X_test)):
    y_pred = y_pred + [1]
y_pred = np.array(y_pred)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     32264
           1       0.70      1.00      0.83     76181

    accuracy                           0.70    108445
   macro avg       0.35      0.50      0.41    108445
weighted avg       0.49      0.70      0.58    108445

Specificity: 0.00
ROC-AUC: 0.5



c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**School Metro & Poverty Level**

Baseline

In [7]:
features = ['school_metro','poverty_level']

In [8]:
print('With Outliers')

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data[features])

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data['fully_funded'], test_size=0.2, random_state=42,stratify=data['fully_funded'])

nb = GaussianNB()

nb.fit(X_train,y_train)

y_pred = nb.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

With Outliers
              precision    recall  f1-score   support

           0       0.38      0.28      0.32     32264
           1       0.73      0.81      0.77     76181

    accuracy                           0.65    108445
   macro avg       0.55      0.54      0.54    108445
weighted avg       0.62      0.65      0.63    108445

Specificity: 0.28
ROC-AUC: 0.5436192485230333



In [9]:
print('Without Outliers')

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

nb = GaussianNB()

nb.fit(X_train,y_train)

y_pred = nb.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

Without Outliers
              precision    recall  f1-score   support

           0       0.37      0.28      0.32     27674
           1       0.74      0.81      0.77     69447

    accuracy                           0.66     97121
   macro avg       0.55      0.55      0.55     97121
weighted avg       0.63      0.66      0.64     97121

Specificity: 0.28
ROC-AUC: 0.5451765561050336



Handling unbalance classes with Random Under Sampler

In [10]:
from imblearn.under_sampling import RandomUnderSampler

In [11]:
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


In [12]:
nb_rus = GaussianNB()
nb_rus.fit(X_train_rus,y_train_rus)
y_pred_rus = nb_rus.predict(X_test)

print('Without Outliers, Random Under Sampling')
print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

Without Outliers, Random Under Sampling
              precision    recall  f1-score   support

           0       0.35      0.42      0.38     27674
           1       0.75      0.69      0.72     69447

    accuracy                           0.61     97121
   macro avg       0.55      0.55      0.55     97121
weighted avg       0.63      0.61      0.62     97121

Specificity: 0.42
ROC-AUC: 0.5451765561050336



In [13]:
#checking distribution

print(f"\nBefore Undersampling - Training examples: {len(X_train)}")
print(f"Class distribution: {np.bincount(y_train)}")

print(f"\nAfter Undersampling - Training examples: {len(X_train_rus)}")
print(f"Class distribution: {np.bincount(y_train_rus)}")


Before Undersampling - Training examples: 388483
Class distribution: [110694 277789]

After Undersampling - Training examples: 221388
Class distribution: [110694 110694]


**Add Resource Type and Primary Subject**

Baseline

In [14]:
features = ['school_metro','poverty_level','resource_type','primary_focus_subject']

In [15]:
#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

nb = GaussianNB()
nb.fit(X_train,y_train)

#testing

y_pred = nb.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

              precision    recall  f1-score   support

           0       0.35      0.58      0.44     27674
           1       0.77      0.57      0.65     69447

    accuracy                           0.57     97121
   macro avg       0.56      0.57      0.54     97121
weighted avg       0.65      0.57      0.59     97121

Specificity: 0.58
ROC-AUC: 0.5733911709690191



With RUS

In [16]:
#training

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

nb_rus = GaussianNB()
nb_rus.fit(X_train_rus,y_train_rus)
y_pred_rus = nb_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.32      0.79      0.45     27674
           1       0.79      0.31      0.45     69447

    accuracy                           0.45     97121
   macro avg       0.55      0.55      0.45     97121
weighted avg       0.66      0.45      0.45     97121

Specificity: 0.79
ROC-AUC: 0.5733911709690191



**Add Students Reached + Funding Request Amt**

Baseline

In [17]:
features = ['school_metro','poverty_level','resource_type','primary_focus_subject']

In [18]:
#without removing outliers

#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data[features])

features_encoded = features_encoded.tolist()

for i in range(len(features_encoded)):
    features_encoded[i] = features_encoded[i] + [data.loc[i,'students_reached']] + [data.loc[i,'total_price_excluding_optional_support']]


features_encoded = np.array(features_encoded)

#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data['fully_funded'], test_size=0.2, random_state=42,stratify=data['fully_funded'])

nb = GaussianNB()
nb.fit(X_train,y_train)

#testing

y_pred = nb.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

              precision    recall  f1-score   support

           0       0.44      0.10      0.16     32264
           1       0.71      0.95      0.81     76181

    accuracy                           0.69    108445
   macro avg       0.58      0.52      0.49    108445
weighted avg       0.63      0.69      0.62    108445

Specificity: 0.10
ROC-AUC: 0.5236995267590181



In [19]:
#without outliers

#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

features_encoded = features_encoded.tolist()

for i in range(len(features_encoded)):
    features_encoded[i] = features_encoded[i] + [data_noutliers.loc[i,'students_reached_scaled']] + [data_noutliers.loc[i,'total_price_excluding_optional_support_scaled']]


features_encoded = np.array(features_encoded)

#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

nb = GaussianNB()
nb.fit(X_train,y_train)

#testing

y_pred = nb.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

              precision    recall  f1-score   support

           0       0.37      0.60      0.46     27674
           1       0.79      0.59      0.68     69447

    accuracy                           0.60     97121
   macro avg       0.58      0.60      0.57     97121
weighted avg       0.67      0.60      0.61     97121

Specificity: 0.60
ROC-AUC: 0.5964008337169351



With RUS

In [20]:
#training

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

nb_rus = GaussianNB()
nb_rus.fit(X_train_rus,y_train_rus)
y_pred_rus = nb_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.33      0.79      0.47     27674
           1       0.82      0.38      0.51     69447

    accuracy                           0.49     97121
   macro avg       0.58      0.58      0.49     97121
weighted avg       0.68      0.49      0.50     97121

Specificity: 0.79
ROC-AUC: 0.5964008337169351



**Add Matching Statuses**

In [25]:
features = ['eligible_double_your_impact_match','eligible_almost_home_match','primary_focus_subject','resource_type','school_metro','poverty_level']

In [26]:
print('Training with Outliers')

#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data[features])

features_encoded = features_encoded.tolist()

for i in range(len(features_encoded)):
    features_encoded[i] = features_encoded[i] + [data.loc[i,'students_reached']] + [data.loc[i,'total_price_excluding_optional_support']]


features_encoded = np.array(features_encoded)

#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data['fully_funded'], test_size=0.2, random_state=42,stratify=data['fully_funded'])

nb = GaussianNB()
nb.fit(X_train,y_train)

#testing

y_pred = nb.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

Training with Outliers
              precision    recall  f1-score   support

           0       0.45      0.15      0.22     32264
           1       0.72      0.92      0.81     76181

    accuracy                           0.69    108445
   macro avg       0.58      0.54      0.52    108445
weighted avg       0.64      0.69      0.63    108445

Specificity: 0.15
ROC-AUC: 0.535664553499056



In [27]:
print('Training without Outliers')

#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

features_encoded = features_encoded.tolist()

for i in range(len(features_encoded)):
    features_encoded[i] = features_encoded[i] + [data_noutliers.loc[i,'students_reached_scaled']] + [data_noutliers.loc[i,'total_price_excluding_optional_support_scaled']]


features_encoded = np.array(features_encoded)

#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

nb = GaussianNB()
nb.fit(X_train,y_train)

#testing

y_pred = nb.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

Training without Outliers
              precision    recall  f1-score   support

           0       0.37      0.67      0.47     27674
           1       0.80      0.54      0.64     69447

    accuracy                           0.58     97121
   macro avg       0.58      0.60      0.56     97121
weighted avg       0.68      0.58      0.60     97121

Specificity: 0.67
ROC-AUC: 0.6033753948025993



In [30]:
#RUS
#training

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

nb_rus = GaussianNB()
nb_rus.fit(X_train_rus,y_train_rus)
y_pred_rus = nb_rus.predict(X_test)

print('Undersampling without Outliers')

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Undersampling without Outliers
              precision    recall  f1-score   support

           0       0.33      0.79      0.46     27674
           1       0.81      0.35      0.49     69447

    accuracy                           0.48     97121
   macro avg       0.57      0.57      0.48     97121
weighted avg       0.67      0.48      0.48     97121

Specificity: 0.79
ROC-AUC: 0.6033753948025993

